# 07 — Mental Models & Observations

This notebook covers Hindsight's two higher-order memory types:
- **Observations**: Auto-consolidated knowledge from multiple facts
- **Mental Models**: User/agent-curated knowledge (highest priority)
- How consolidation works
- How to create and manage mental models
- The knowledge hierarchy in action

In [ ]:
from hindsight_client import Hindsight
from datetime import datetime, timezone

HINDSIGHT_URL = "http://localhost:8888"
BANK_ID = "tutorial-mental-models"

client = Hindsight(base_url=HINDSIGHT_URL)

# Create a bank that's good at learning
client.create_bank(
    bank_id=BANK_ID,
    name="Mental Models Tutorial",
    enable_observations=True,
    reflect_mission="""
    I am a learning assistant. I track user preferences and build
    an increasingly accurate model of the user over time.
    """
)
print(f"✓ Bank '{BANK_ID}' ready")

## 1. Observations: How They Emerge

Feed Hindsight multiple related facts and watch observations form.

In [ ]:
# Simulate learning about a user's coding preferences over time
# Week 1: Initial impressions
week1_facts = [
    "User prefers Python for all backend work",
    "User uses type hints in all Python code",
    "User prefers functional programming patterns over OOP",
    "User uses pytest for testing with 90%+ coverage target",
]

for fact in week1_facts:
    client.retain(
        bank_id=BANK_ID,
        content=fact,
        context="coding-preference",
        timestamp=datetime(2026, 1, 15, tzinfo=timezone.utc).isoformat()
    )
    print(f"✓ Week 1: {fact}")

print("\nHindsight will auto-consolidate these into observations...")

In [ ]:
# Week 2: More evidence strengthens observations
week2_facts = [
    "User chose FastAPI (Python) over Express (Node.js) for the new API",
    "User rejected a PR that used classes, asked for dataclasses instead",
    "User added mypy strict mode to CI pipeline",
]

for fact in week2_facts:
    client.retain(
        bank_id=BANK_ID,
        content=fact,
        context="coding-preference",
        timestamp=datetime(2026, 2, 1, tzinfo=timezone.utc).isoformat()
    )
    print(f"✓ Week 2: {fact}")

print("\nObservations strengthen with each new fact...")

In [ ]:
# Week 3: A SURPRISE — user's preferences shift
week3_facts = [
    "User is exploring Rust for a performance-critical service",
    "User mentioned they're tired of Python's GIL limitations",
    "User started a side project in Rust for a real-time data processor",
]

for fact in week3_facts:
    client.retain(
        bank_id=BANK_ID,
        content=fact,
        context="coding-preference",
        timestamp=datetime(2026, 3, 1, tzinfo=timezone.utc).isoformat()
    )
    print(f"✓ Week 3: {fact}")

print("\nObservations UPDATE (not overwrite) — history is preserved!")

## 2. See the Observation Evolution

Query Hindsight to see how its understanding has evolved.

In [ ]:
# Check what observations have formed
stats = client.get_bank_stats(bank_id=BANK_ID)
print(f"Total memories: {stats.get('total_memories', '?')}")
print(f"Observations: {stats.get('observations', '?')}")
print(f"World facts: {stats.get('world_facts', '?')}")

# Recalling observations
results = client.recall(
    bank_id=BANK_ID,
    query="user's coding preferences",
    types=["observation"],
    budget="high"
)

print(f"\nObservation results: {len(results.results)}")
for r in results.results:
    print(f"  [{r.type}] {r.text}")

In [ ]:
# Reflect: Hindsight considers ALL evidence (old + new)
answer = client.reflect(
    bank_id=BANK_ID,
    query="Based on everything you know, recommend a language for a new high-performance service."
)
print("=== Hindsight's Recommendation ===")
print(answer.text)
print(f"\nBased on {len(answer.based_on)} sources")

## 3. Mental Models: Curated Knowledge

Mental models are explicitly created (not auto-generated). They have the highest
priority during `reflect()`.

In [ ]:
# Create a mental model — team coding standards
model = client.create_mental_model(
    bank_id=BANK_ID,
    name="Team Python Standards v2",
    content="""
    ## Team Python Coding Standards (updated June 2026)
    
    ### Type System
    - All function signatures MUST have type hints
    - Use mypy strict mode in CI
    - Prefer dataclasses over plain dicts
    
    ### Style
    - Format with ruff, line length 100
    - Google-style docstrings
    - Max function length: 50 lines
    
    ### Testing
    - pytest with pytest-asyncio
    - Minimum 85% coverage
    - Property-based testing with Hypothesis for critical paths
    
    ### Async
    - Use asyncio (not trio)
    - asyncio.run() at entry points only
    - httpx for async HTTP, not requests
    
    ### Architecture
    - Clean architecture pattern
    - Repository pattern for data access
    - Dependency injection via FastAPI's Depends
    """,
    tags=["standards", "python", "team"]
)
print(f"✓ Created mental model: Team Python Standards v2")
print(f"  Model ID: {model.id if hasattr(model, 'id') else 'N/A'}")

In [ ]:
# List all mental models
models = client.list_mental_models(bank_id=BANK_ID)

print(f"Mental models: {len(models) if isinstance(models, list) else 'N/A'}")
if isinstance(models, list):
    for model in models:
        m_name = getattr(model, 'name', str(model))
        m_id = getattr(model, 'id', '?')
        m_tags = getattr(model, 'tags', [])
        print(f"  • {m_name} (id={m_id})")
        if m_tags:
            print(f"    Tags: {', '.join(m_tags)}")

## 4. Mental Model Priority — It Overrides Everything

Even if there are 20 raw facts about Python, the mental model wins.

In [ ]:
# Even though we have facts about the user exploring Rust...
# The mental model says:
#   - "All function signatures MUST have type hints"
#   - "Use asyncio (not trio)"
#   - "Clean architecture pattern"

answer = client.reflect(
    bank_id=BANK_ID,
    query="For a new FastAPI project, should I use asyncio or trio for async?"
)

print("=== Hindsight's Answer ===")
print(answer.text[:400])
print("\n(Note: The mental model 'Team Python Standards v2' takes priority")

In [ ]:
# Exclude mental models to get analysis from raw facts only
answer_no_mm = client.reflect(
    bank_id=BANK_ID,
    query="For a new project, should I prioritize Rust or Python?",
    exclude_mental_models=True  # ← Ignore mental models
)

print("=== Without Mental Models (raw facts only) ===")
print(answer_no_mm.text[:400])

print("\n" + "="*60)

answer_with_mm = client.reflect(
    bank_id=BANK_ID,
    query="For a new project, should I prioritize Rust or Python?",
    exclude_mental_models=False
)

print("=== With Mental Models ===")
print(answer_with_mm.text[:400])

## 5. Updating a Mental Model

In [ ]:
# Get the model ID first
models = client.list_mental_models(bank_id=BANK_ID)
if isinstance(models, list) and models:
    model = models[0]
    model_id = getattr(model, 'id', None)
    
    if model_id:
        # Update the mental model
        client.update_mental_model(
            bank_id=BANK_ID,
            mental_model_id=model_id,
            content="""
            ## Team Python Standards v2.1 (July 2026)
            
            ### UPDATES:
            - Line length increased to 120 (team vote)
            - Added: Pydantic v2 for all data validation
            - Added: structlog for structured logging
            
            ### Type System
            - All function signatures MUST have type hints
            - Use mypy strict mode in CI
            - Prefer Pydantic v2 models over dataclasses for API data
            
            ### Style
            - Format with ruff, line length 120 (updated)
            - Google-style docstrings
            - Use structlog for logging
            
            ### Architecture
            - Clean architecture pattern
            - Repository pattern for data access
            - Pydantic v2 for all data validation
            """,
            tags=["standards", "python", "team", "v2.1"]
        )
        print(f"✓ Updated mental model {model_id} to v2.1")

## 6. The Knowledge Hierarchy in Action

In [ ]:
# Demonstrate the priority chain: Mental Models > Observations > Raw Facts
print("=== Knowledge Hierarchy ===\n")

print("1. MENTAL MODELS (highest priority)")
models = client.list_mental_models(bank_id=BANK_ID)
if isinstance(models, list):
    for m in models:
        print(f"   • {getattr(m, 'name', str(m))}")

print("\n2. OBSERVATIONS (auto-consolidated)")
obs = client.recall(bank_id=BANK_ID, query="preferences", types=["observation"], budget="low")
for r in obs.results[:3]:
    print(f"   • {r.text[:100]}...")

print("\n3. RAW FACTS (world + experience)")
facts = client.recall(bank_id=BANK_ID, query="preferences", types=["world"], budget="low")
for r in facts.results[:5]:
    print(f"   • {r.text[:100]}...")

print("\nDuring reflect(), Hindsight checks in order:")
print("  Mental Models → Observations → Raw Facts")

## 7. Practical Pattern: Project Onboarding via Mental Models

In [ ]:
def onboard_project(client, bank_id, project_name, context_dict):
    """Create a mental model for project onboarding."""
    
    content_parts = [f"# {project_name} — Project Context\n"]
    
    for section, items in context_dict.items():
        content_parts.append(f"\n## {section}")
        for item in items:
            content_parts.append(f"- {item}")
    
    content = "\n".join(content_parts)
    
    model = client.create_mental_model(
        bank_id=bank_id,
        name=f"Project Context: {project_name}",
        content=content,
        tags=["onboarding", f"project:{project_name.lower()}"]
    )
    
    return model

# Usage
context = {
    "Tech Stack": [
        "Backend: FastAPI + SQLAlchemy 2.0",
        "Database: PostgreSQL 17 + pgvector",
        "Cache: Redis (ElastiCache)",
        "Infra: AWS EKS + Terraform",
        "CI/CD: GitHub Actions + ArgoCD",
    ],
    "Architecture Decisions": [
        "Event-driven microservices via NATS",
        "CQRS pattern for read/write separation",
        "Saga pattern for distributed transactions",
    ],
    "Known Pitfalls": [
        "Order service has tight coupling with legacy DB",
        "Cold start latency on EKS Fargate (~30s)",
        "NATS message size limit: 1MB (use S3 for larger payloads)",
    ],
    "Team Contacts": [
        "Alice Chen — Backend Lead",
        "Bob Martinez — Frontend Lead",
        "Carol Wu — DevOps / Platform",
    ],
}

onboard_project(client, BANK_ID, "Project Phoenix", context)
print("✓ Created project onboarding mental model")

## 8. Cleanup

In [ ]:
# client.delete_bank(bank_id=BANK_ID)
print("Bank preserved.")

## Summary

You learned:
1. **Observations** — how they auto-consolidate from multiple related facts
2. **Evolution tracking** — observations update (not overwrite), history preserved
3. **Mental models** — creating, listing, updating curated knowledge
4. **Priority** — mental models take highest priority during reflect()
5. **Excluding models** — `exclude_mental_models=True` for raw-fact analysis
6. **Knowledge hierarchy** — Mental Models → Observations → Raw Facts
7. **Onboarding pattern** — using mental models for project context

**Next:** [08_async_batch.ipynb](08_async_batch.ipynb) — high-throughput patterns